## Environment Setup & Data Ingestion

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, OrdinalEncoder

# Suppress non-critical warnings & set random seed
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

# Ingest datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print(f"Loaded Train Data Shape: {train_df.shape} (Expected 12,000, 35)")
print(f"Loaded Test Data Shape:  {test_df.shape} (Expected 8,000, 34)")

Loaded Train Data Shape: (12000, 35) (Expected 12,000, 35)
Loaded Test Data Shape:  (8000, 34) (Expected 8,000, 34)


## Financial & Tenure Interaction Features

In [2]:
def create_financial_features(df):
    df = df.copy()

    # 1. Effective Monthly Charge after applied discount
    df['effective_monthly_charge'] = df['monthly_charge'] * (1.0 - df['discount_percentage'] / 100.0)

    # 2. Historical Monthly Average Spend over customer tenure
    df['historical_monthly_avg'] = df['lifetime_spend'] / (df['tenure_months'] + 1.0)

    # 3. Spend Discrepancy (Current price vs historical average spend)
    df['spend_discrepancy'] = df['effective_monthly_charge'] - df['historical_monthly_avg']

    # 4. Total Discount Savings Amount ($)
    df['discount_savings_amount'] = df['monthly_charge'] * (df['discount_percentage'] / 100.0)

    # 5. Cost per Service Unit
    df['cost_per_service'] = df['effective_monthly_charge'] / (df['number_of_services'] + 1.0)

    return df

train_df = create_financial_features(train_df)
test_df = create_financial_features(test_df)

print("Financial interaction features constructed successfully.")

Financial interaction features constructed successfully.


## Engagement & Usage Intensity Ratios

In [3]:
def create_engagement_features(df):
    df = df.copy()

    # 1. Data Usage Density per Active Service
    df['data_per_service'] = df['data_usage_gb'] / (df['number_of_services'] + 1.0)

    # 2. Streaming Intensity (Streaming hours per GB consumed)
    df['streaming_intensity'] = df['streaming_hours_monthly'] / (df['data_usage_gb'] + 1.0)

    # 3. Weekly App Engagement Time (Total Minutes)
    df['weekly_app_minutes'] = df['app_sessions_weekly'] * df['average_session_minutes']

    # 4. Monthly App Usage Hours
    df['monthly_app_hours'] = (df['weekly_app_minutes'] * 4.0) / 60.0

    # 5. Device Age Ratio relative to Customer Tenure
    df['device_age_tenure_ratio'] = df['device_age_months'] / (df['tenure_months'] + 1.0)

    return df

train_df = create_engagement_features(train_df)
test_df = create_engagement_features(test_df)

print("Engagement and usage intensity ratios generated successfully.")

Engagement and usage intensity ratios generated successfully.


## Customer Friction & Support Severity Indices

In [4]:
def create_friction_features(df):
    df = df.copy()

    # 1. Friction Severity Index (Weighted sum of complaints, calls, and payment delays)
    df['friction_score'] = (df['network_complaints'] * 2.0) + df['support_calls_90d'] + df['payment_delay_days']

    # 2. Network Complaints Rate per Tenure Month
    df['complaints_per_tenure'] = df['network_complaints'] / (df['tenure_months'] + 1.0)

    # 3. Support Call Frequency per Active Service
    df['calls_per_service'] = df['support_calls_90d'] / (df['number_of_services'] + 1.0)

    # 4. Delay Severity Ratio (Delay days per late payment instance)
    df['delay_per_late_event'] = df['payment_delay_days'] / (df['late_payment_count'] + 1.0)

    # 5. Dissatisfaction Compound Score
    df['dissatisfaction_call_compound'] = (10.0 - df['satisfaction_score']) * (df['support_calls_90d'] + 1.0)

    return df

train_df = create_friction_features(train_df)
test_df = create_friction_features(test_df)

print("Friction severity indices successfully generated.")

Friction severity indices successfully generated.


## High-Risk Categorical & Behavioral Flags

In [5]:
def create_risk_flags(df):
    df = df.copy()

    # 1. High Friction Month-to-Month Customer Flag
    df['flag_high_friction_m2m'] = (
        (df['contract_type'] == 'Month-to-month') &
        (df['payment_delay_days'] > 5)
    ).astype(int)

    # 2. Digital Native Profile (Autopay + Paperless Billing)
    df['flag_digital_native'] = (
        (df['autopay'].astype(str).str.lower().isin(['1', 'true', 'yes'])) &
        (df['paperless_billing'].astype(str).str.lower().isin(['1', 'true', 'yes']))
    ).astype(int)

    # 3. High-Cost Single Service Customer Risk
    df['flag_high_cost_single_service'] = (
        (df['number_of_services'] == 1) &
        (df['monthly_charge'] > df['monthly_charge'].median())
    ).astype(int)

    # 4. Severe Dissatisfaction Flag
    df['flag_severe_dissatisfaction'] = (
        (df['satisfaction_score'] <= 3) &
        (df['support_calls_90d'] >= 3)
    ).astype(int)

    return df

train_df = create_risk_flags(train_df)
test_df = create_risk_flags(test_df)

print("High-risk behavioral flags generated successfully.")

High-risk behavioral flags generated successfully.


## Group-Level Cohort Aggregations (Target-Independent)

In [7]:
group_cols = ['customer_segment', 'plan_tier']
target_agg_cols = ['monthly_charge', 'data_usage_gb', 'satisfaction_score']

# Compute cohort summary statistics strictly on the training set to prevent leakage
group_stats = train_df.groupby(group_cols)[target_agg_cols].agg(['mean', 'std']).reset_index()

# Flatten multi-index column headers
group_stats.columns = [
    c[0] if c[1] == '' else f"group_{c[0]}_{c[1]}"
    for c in group_stats.columns
]

# Merge peer benchmarks into train and test sets
train_df = train_df.merge(group_stats, on=group_cols, how='left')
test_df = test_df.merge(group_stats, on=group_cols, how='left')

# Calculate customer deviations relative to peer benchmarks
for col in target_agg_cols:
    train_df[f"{col}_diff_from_group_mean"] = train_df[col] - train_df[f"group_{col}_mean"]
    test_df[f"{col}_diff_from_group_mean"] = test_df[col] - test_df[f"group_{col}_mean"]

print("Group cohort aggregations and relative deviations created.")

Group cohort aggregations and relative deviations created.


## Out-of-Fold (OOF) Target Encoding (Leak-Free)

In [8]:
TARGET_ENC_COLS = ['acquisition_channel', 'region', 'promotion_type', 'internet_type']
TARGET_COL = 'churn'

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
global_mean = train_df[TARGET_COL].mean()

# Initialize encoding columns
for col in TARGET_ENC_COLS:
    train_df[f"{col}_te"] = np.nan
    test_df[f"{col}_te"] = 0.0

# 5-Fold Out-Of-Fold calculation for training data
for train_idx, val_idx in skf.split(train_df, train_df[TARGET_COL]):
    tr_fold = train_df.iloc[train_idx]

    for col in TARGET_ENC_COLS:
        means = tr_fold.groupby(col)[TARGET_COL].mean()
        train_df.iloc[val_idx, train_df.columns.get_loc(f"{col}_te")] = (
            train_df.iloc[val_idx][col].map(means).fillna(global_mean)
        )

# Apply global training encodings to test data
for col in TARGET_ENC_COLS:
    full_means = train_df.groupby(col)[TARGET_COL].mean()
    test_df[f"{col}_te"] = test_df[col].map(full_means).fillna(global_mean)

print("Leak-free Out-of-Fold target encodings calculated successfully.")

Leak-free Out-of-Fold target encodings calculated successfully.


## Categorical Ordinal & Frequency Encoding

In [9]:
CATEGORICAL_COLS = [
    "customer_segment", "region", "acquisition_channel", "contract_type",
    "plan_tier", "internet_type", "device_type", "support_plan",
    "payment_method", "promotion_type", "streaming_service",
    "priority_support", "autopay", "paperless_billing", "family_plan",
    "international_plan"
]

# Frequency Encoding
for col in CATEGORICAL_COLS:
    freq_map = train_df[col].value_counts(normalize=True)
    train_df[f"{col}_freq"] = train_df[col].map(freq_map)
    test_df[f"{col}_freq"] = test_df[col].map(freq_map).fillna(0.0)

# Ordinal Encoding
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_df[CATEGORICAL_COLS] = encoder.fit_transform(train_df[CATEGORICAL_COLS].astype(str))
test_df[CATEGORICAL_COLS] = encoder.transform(test_df[CATEGORICAL_COLS].astype(str))

print("Categorical ordinal and frequency encodings complete.")

Categorical ordinal and frequency encodings complete.


## Skewness Handling (Log Transformations) & Robust Scaling

In [10]:
SKEWED_COLS = ['lifetime_spend', 'payment_delay_days', 'data_usage_gb', 'streaming_hours_monthly']

# Log1p transformation for skewed attributes
for col in SKEWED_COLS:
    if col in train_df.columns:
        train_df[f"{col}_log"] = np.log1p(np.maximum(0, train_df[col]))
        test_df[f"{col}_log"] = np.log1p(np.maximum(0, test_df[col]))

# Isolate feature matrix columns
EXCLUDE_COLS = ['customer_id', 'churn']
FEATURE_COLS = [col for col in train_df.columns if col not in EXCLUDE_COLS]

# Impute any residual nulls with medians before scaling
train_df[FEATURE_COLS] = train_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median())
test_df[FEATURE_COLS] = test_df[FEATURE_COLS].fillna(train_df[FEATURE_COLS].median())

# Fit RobustScaler on training set
scaler = RobustScaler()
train_scaled = train_df.copy()
test_scaled = test_df.copy()

train_scaled[FEATURE_COLS] = scaler.fit_transform(train_df[FEATURE_COLS])
test_scaled[FEATURE_COLS] = scaler.transform(test_df[FEATURE_COLS])

print(f"Log transformations applied and {len(FEATURE_COLS)} features scaled.")

Log transformations applied and 85 features scaled.


## Feature Integrity & Schema Verification

In [11]:
# Verification assertions
assert train_scaled[FEATURE_COLS].isnull().sum().sum() == 0, "Null values present in train feature matrix!"
assert test_scaled[FEATURE_COLS].isnull().sum().sum() == 0, "Null values present in test feature matrix!"
assert set(FEATURE_COLS) == set(test_scaled.columns) - {'customer_id'}, "Feature column mismatch between train and test!"

print("=" * 55)
print("     FEATURE ENGINEERING INTEGRITY AUDIT REPORT     ")
print("=" * 55)
print(f"Baseline Feature Count:    33")
print(f"Engineered Feature Count:  {len(FEATURE_COLS)}")
print(f"Train Dataset Dimensions:  {train_scaled.shape}")
print(f"Test Dataset Dimensions:   {test_scaled.shape}")
print("=" * 55)
print("Feature schema successfully validated.")

     FEATURE ENGINEERING INTEGRITY AUDIT REPORT     
Baseline Feature Count:    33
Engineered Feature Count:  85
Train Dataset Dimensions:  (12000, 87)
Test Dataset Dimensions:   (8000, 86)
Feature schema successfully validated.


## Dataset Export

In [12]:
TRAIN_OUTPUT_PATH = "train_engineered.csv"
TEST_OUTPUT_PATH = "test_engineered.csv"

train_scaled.to_csv(TRAIN_OUTPUT_PATH, index=False)
test_scaled.to_csv(TEST_OUTPUT_PATH, index=False)

print(f"Engineered train matrix saved to: '{TRAIN_OUTPUT_PATH}'")
print(f"Engineered test matrix saved to:  '{TEST_OUTPUT_PATH}'")

Engineered train matrix saved to: 'train_engineered.csv'
Engineered test matrix saved to:  'test_engineered.csv'
